## Time Series ACF Lag Drift Monitor

In [0]:
%pip install /Volumes/mlcore_dev/mlcore_init_scripts/mlworkspace/MLCORE_INIT/monitor_db_uc/MLCoreSDK_monitor_db_uc-0.4.6-py3-none-any.whl --force-reinstall
# %pip install /Volumes/mlcore_dev/mlcore_init_scripts/mlworkspace/MLCORE_INIT/monitor_db_uc/MonitorReportKit_monitor_db_uc-1.0.5-py3-none-any.whl --force-reinstall
# %pip install /dbfs/FileStore/custom_packages/MonitorReportKit-0.5.31-py3-none-any.whl

<b>Imports

Along with the imports required for the notebook to execute custom transformations, we have to import <b>MLCoreClient</b> from <b>MLCORE_SDK</b>, which provides helper methods to integrate the custom notebook with rest of the Data Prep or Data Prep Deployment flow.

In [0]:
from functools import reduce
from pyspark.sql import DataFrame
import seaborn as sns
from pyspark.sql import functions as F, types as T

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
# from monitoring.data.feature_drift.batch_monitors import monitor_feature_drift_in_memory

import numpy as np
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tools.eval_measures import rmse
from statsmodels.tsa.stattools import acf
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType, IntegerType

In [0]:
# from utils import utils
import time
from delta.tables import *
import pandas as pd
import numpy as np
from pyspark.sql import (
    types as T,
    functions as F,
    Window
)
from datetime import datetime
from pyspark.sql.types import StructType
from pyspark.sql.window import Window
from datetime import datetime, timedelta
import ast
import json

In [0]:
# %pip install tqdm mlflow
# from MLCORE_SDK import mlclient

<b> Reading parent data for the notebook

<b>read_parent_data</b> is a method which reads the outputs of parent nodes selected for the "Notebook" transformer. <b>df_dict</b> is a dictionary, the keys are the task keys and the values are the dataframes for the respective task keys.

e.g. If Notebook transformer is a child of task1 and task2, then df_dict will look something like this:

    {
      "task1": task1_DataFrame,
      "task2": task2_DataFrame
    }

In [0]:
df_dict = mlclient.log(operation_type = "register_plots",
    dbutils = dbutils,
    spark = spark,
    request_type="read_parent_data")

<b> Applying desired transformation

Now, we can apply the desired transformation on the loaded data. Here, we are performing union on all the parent DataFrames.

In [0]:
df_final = reduce(DataFrame.unionAll, df_dict.values())
df_pandas = df_final.toPandas()

<b> EDA on the resultant data

We can peform Exploratory Data Analysis on the resultant or source data, if required. The generated plots can be saved using the <b>push_plots_to_mlcore</b> method available in <b>MLCORE_SDK</b>. The generated plots then can be found under the respective Notebook transformer node's plots.

In [0]:
df = spark.read.load('dbfs:/mnt/mlcore/dev/034683915d3a4de498bbe972dfaa3c8d/sales_monitor/64f6fee9559febf3c972244f/PREVIEW/task1')

In [0]:
#dbutils
jobrunstr = str(dbutils.widgets.get("job_run_str"))
az_container_name = dbutils.widgets.get("az_container_name")
dev = dbutils.widgets.get("env")
env = dbutils.widgets.get("env")
mode = dbutils.widgets.get("mode")
project_id = dbutils.widgets.get("project_id")
version = dbutils.widgets.get("version")
task_key = dbutils.widgets.get("task_key")
batch_size = int(dbutils.widgets.get("batch_size"))

if mode == 'train':
    job_id = jobrunstr.split('/')[0]
    run_id = jobrunstr.split('/')[1]
if mode == 'preview':
    job_id = jobrunstr.split('/')[0]
    run_id = jobrunstr.split('/')[1]

else:
    job_id = jobrunstr
    run_id = dbutils.widgets.get("run_id")

In [0]:
def select_store_data(df,store_id,no_of_items):
    final_df=df[ (df['store_id'] ==store_id) & (df['item_id'] ==1) ]
    final_df["item_id_"+str(1)+"_sales"]=final_df["sales"]

    for i_id in  range(2,no_of_items+1):

        filtered_df = df[ (df['store_id'] ==store_id) & (df['item_id'] ==i_id) ]
        final_df["item_id_"+str(i_id)+"_sales"]= filtered_df["sales"].values
    final_df=final_df.drop(["sales","item_id"],axis=1)
    final_df=final_df.reset_index()
    final_df=final_df.drop(["index"],axis=1)
    
    return final_df    

### Method to monitor your data for drift

In [0]:
def monitor_data_drift(df,ref_year,ref_month,curr_year,curr_months,monitored_features,monitoring_algos,date_column):
    
    drift_list=[]

    bn=0
    df[date_column] = pd.to_datetime(df[date_column]) 
    df['year'] = df[date_column].dt.year
    df['month'] = df[date_column].dt.month

    for curr_month in [curr_months]:
        
        ref_df = df[ (df['year'] == ref_year) & (df['month'] == curr_month) ]

        curr_df = df[(df['year'] == curr_year) & (df['month'] == curr_month)]

        result=monitor_feature_drift_in_memory(
            data_type= "timeseries",
            reference_df=ref_df,
            current_df=curr_df,
            monitored_features=monitored_features,
            monitoring_algos= monitoring_algos,
            preprocess= False,
            date_column=date_column,
            monitoring_algo_thresholds= None,)
        
        print("Batch_No:"+str(bn))
        bn=bn+1
        drift_list.append(result)

    df = df.drop(['year', 'month'], axis=1)
    return drift_list


### Sample methods to fetch drifted store X items

In [0]:
def get_item_drift(drift_list,monitored_features,Batch_No,Current_Data_MONTH,Current_Data_YEAR):

    saved_list=[]

    drifted_features_with_store_no={}
    for fno in range(len(monitored_features)):

        drifted_features_with_store_no[monitored_features[fno]]=[]

    for i,v in enumerate(drift_list):
        # print("Store_No:",i+1)
        store_drift_status=drift_list[i][0][0]
        isd_list=[]
        drifted_features=[]
        weighted_is_drift_list=[]
        
        for feature_no in range(len(monitored_features)):
            
            final_is_drift=store_drift_status["is_drift"][feature_no]

            weighted_is_drift=store_drift_status[monitored_features[feature_no]]["weighted_deviation"]

            if final_is_drift==1:
                drifted_features.append(monitored_features[feature_no])
                drifted_features_with_store_no[monitored_features[feature_no]].append(i+1)

            isd_list.append(final_is_drift)
            weighted_is_drift_list.append(weighted_is_drift)

        saved_list.append([ Batch_No,Current_Data_MONTH,Current_Data_YEAR ]+[i+1]+[isd_list]+[drifted_features]+weighted_is_drift_list)
    drift_df=pd.DataFrame(saved_list)

    drift_df_columns=["Batch_No","Current_Data_MONTH","Current_Data_YEAR","Store_No","feature_is_drift_list","drifted_features"]

    for feature_no in range(len(monitored_features)):

        drift_df_columns.append(monitored_features[feature_no]+"_weighted_is_drift")
    drift_df.columns=drift_df_columns


    return drift_df,drifted_features_with_store_no

def get_item_drifted_with_store_no(df):
    
    item_drifted_with_store_no=[]

    for store_no in df["Store_No"]:
    
        filtered_df = df.loc[df['Store_No'] == store_no, ['Store_No', 'drifted_features']]
    

        list_of_features_drifted=list(filtered_df["drifted_features"])[0] 
    
    
        for feature_drifted in list_of_features_drifted:

            # print(feature_drifted)

            item_drifted_with_store_no.append([store_no,feature_drifted])

    return item_drifted_with_store_no

### Method to plot TS Decomposition plots

In [0]:
def plot_subplots(base_df,ref_year,ref_month,curr_year,curr_month,monitored_features,date_column=None,store_id=None):

    if len(monitored_features) == 0:
        print("Columns not provided for decomposition.")
        return None
    if date_column is None:
        print("Date Column is not provided.")
        return None
    if base_df[date_column].duplicated().any():
        print("The date column of your data has duplicate timestamps.")
        return None

    base_df[date_column] = pd.to_datetime(base_df[date_column])
    base_df_sorted = base_df.sort_values(date_column)
    frequency = pd.infer_freq(
        pd.DatetimeIndex(base_df_sorted[date_column].to_list())
    )

    base_df_sorted = base_df_sorted.set_index(date_column).asfreq(frequency)
    base_df_sorted = base_df_sorted.fillna(method="ffill")
    base_df_sorted = base_df_sorted.reset_index()

    base_df_sorted.loc[:, date_column] = pd.to_datetime(
        base_df_sorted[date_column]
    )

    base_df_sorted['year'] = base_df_sorted[date_column].dt.year
    base_df_sorted['month'] = base_df_sorted[date_column].dt.month

    ref_df = base_df_sorted[ (base_df_sorted['year'] == ref_year) & (base_df_sorted['month'] == ref_month) ]
    curr_df = base_df_sorted[(base_df_sorted['year'] == curr_year) & (base_df_sorted['month'] == curr_month)]


    base_df_sorted = base_df_sorted.set_index(date_column)
    ref_df =ref_df.set_index(date_column)
    curr_df =curr_df.set_index(date_column)


    fig, axes = plt.subplots(4, 2, figsize=(20, 15))

    # Plot reference data
    axes[0, 0].plot(ref_df.index, ref_df[monitored_features].values, label='Reference Data', color='blue')
    axes[0, 0].set_title('Reference Dataset')
    axes[0, 0].set_xlabel('Time')
    axes[0, 0].set_ylabel('Value')
    axes[0, 0].tick_params(axis='x', rotation=45)
    axes[0, 0].legend()

    # Plot current data
    axes[0, 1].plot(curr_df.index, curr_df[monitored_features].values, label='Current Data', color='orange')
    axes[0, 1].set_title('Current Dataset')
    axes[0, 1].set_xlabel('Time')
    axes[0, 1].set_ylabel('Value')
    axes[0, 1].tick_params(axis='x', rotation=45)
    axes[0, 1].legend()

    # Time series decomposition
    components = ['Trend', 'Seasonal', 'Resid']
    for i, component in enumerate(components):
        ref_decomposition = seasonal_decompose(ref_df[monitored_features], model='additive')
        curr_decomposition = seasonal_decompose(curr_df[monitored_features], model='additive')

        ref_component = getattr(ref_decomposition, component.lower())
        curr_component = getattr(curr_decomposition, component.lower())

        row = i + 1
        axes[row, 0].plot(ref_df.index, ref_component, label=component, color='green')
        axes[row, 0].set_title(f'Reference {component}')
        axes[row, 0].set_xlabel('Time')
        axes[row, 0].set_ylabel('Value')
        axes[row, 0].tick_params(axis='x', rotation=45)
        axes[row, 0].legend()

        axes[row, 1].plot(curr_df.index, curr_component, label=component, color='red')
        axes[row, 1].set_title(f'Current {component}')
        axes[row, 1].set_xlabel('Time')
        axes[row, 1].set_ylabel('Value')
        axes[row, 1].tick_params(axis='x', rotation=45)
        axes[row, 1].legend()

    plt.tight_layout()
    plt.show()
    mlclient.log(operation_type = "register_plots",dbutils = dbutils, figure_to_save=fig, 
        plot_name=f'Decomposition_plot_for_Store_{store_id}_Item_{monitored_features[0]}_Year_{curr_year}_Month_{curr_month}', 
        folder_name = 'data_drift/decomposition_plots',
        request_type="push_plot")
    return plt

### Method to plot ACF, PACF plots

In [0]:
def get_acf_df(
    base_df: pd.DataFrame, feature_col: str, lags: List[int] = None
):
    """Function to generate the Auto Correlation Function data.

    Parameters
    ----------
    base_df : pd.DataFrame
        data containing the time series for the given variable.
    feature_col : str
        column name for which the acf needs to be calculated. This should be a `numerical` datatype column.
    lags : List[int], optional
        list of lag values on which the maximum lag is computed on which it calculates the ACF, by default None, by default None

    Returns
    -------
    pd.DataFrame
        A DataFrame with the ACF values computed for the specified lags.

    """
    if lags is None:
        lags = []
    else:
        # filter out the -ve values in lag
        lags = list(filter(lambda lag: True if lag > 0 else False, lags))

    # check for both empty lags or lags is None
    if not lags:
        # nlags = 50
        nlags = min(100, base_df.shape[0] - 1)
    else:
        nlags = max(lags)
        if nlags > (base_df.shape[0] - 1):
            raise ValueError(
                "nlags value("
                + str(nlags)
                + ") cannot be greater than the maximum value("
                + str(base_df.shape[0] - 1)
                + ") that acf can take."
            )
    return pd.DataFrame.from_dict(
        {
            "lags": range(0, nlags + 1),
            # "correlation": acf(data[col], nlags=nlags).round(3),
            "correlation": [
                round(base_df[feature_col].autocorr(lag=i), 3)
                for i in range(0, nlags + 1)
            ],
        }
    ).fillna(0)

In [0]:

def plot_acf_subplots(base_df,ref_year,ref_month,curr_year,curr_month,monitored_features,date_column,store_id=None):
    
    if len(monitored_features) == 0:
        print("Columns not provided for decomposition.")
        return None
    if date_column is None:
        print("Date Column is not provided.")
        return None
    if base_df[date_column].duplicated().any():
        print("The date column of your data has duplicate timestamps.")
        return None

    base_df[date_column] = pd.to_datetime(base_df[date_column])
    base_df_sorted = base_df.sort_values(date_column)
    frequency = pd.infer_freq(
        pd.DatetimeIndex(base_df_sorted[date_column].to_list())
    )

    base_df_sorted = base_df_sorted.set_index(date_column).asfreq(frequency)
    base_df_sorted = base_df_sorted.fillna(method="ffill")
    base_df_sorted = base_df_sorted.reset_index()

    base_df_sorted.loc[:, date_column] = pd.to_datetime(
        base_df_sorted[date_column]
    )

    base_df_sorted['year'] = base_df_sorted[date_column].dt.year
    base_df_sorted['month'] = base_df_sorted[date_column].dt.month

    ref_df = base_df_sorted[ (base_df_sorted['year'] == ref_year) & (base_df_sorted['month'] == ref_month) ]
    curr_df = base_df_sorted[(base_df_sorted['year'] == curr_year) & (base_df_sorted['month'] == curr_month)]


    base_df_sorted = base_df_sorted.set_index(date_column)
    ref_df =ref_df.set_index(date_column)
    curr_df =curr_df.set_index(date_column)

    ref_acf_df=get_acf_df(ref_df,monitored_features[0])
    curr_acf_df=get_acf_df(curr_df,monitored_features[0])

    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6))


    ax1.stem(ref_acf_df['lags'], ref_acf_df['correlation'], use_line_collection=True)
    ax1.set_title('ACF - Reference Data')
    ax1.set_xlabel('Lags')
    ax1.set_ylabel('Correlation')

    
    ax2.stem(curr_acf_df['lags'], curr_acf_df['correlation'], use_line_collection=True)
    ax2.set_title('ACF - Current Data')
    ax2.set_xlabel('Lags')
    ax2.set_ylabel('Correlation')

    
    plt.tight_layout()

    mlclient.log(operation_type = "register_plots",dbutils = dbutils, figure_to_save=fig, 
        plot_name=f'Store_{store_id}_Item_{monitored_features[0]}_Year_{curr_year}_Month_{curr_month}', 
        folder_name = 'data_drift/LAG_drift_Plots',
        request_type="push_plot")
    
    return plt


### Method to plot Monitoring Evaluation plots like KDE, Histogram

In [0]:
def get_kde_plot(df,ref_year,ref_month,curr_year,curr_month,monitored_features,date_column=None,store_id=None):

    df[date_column] = pd.to_datetime(df[date_column]) 

    df['year'] = df[date_column].dt.year
    df['month'] = df[date_column].dt.month

    ref_df = df[(df['year'] == ref_year) & (df['month'] == ref_month) ]
    curr_df = df[(df['year'] == curr_year) & (df['month'] == curr_month) ]

    fig = plt.figure(figsize=(10, 6))

    sns.kdeplot(data=ref_df[monitored_features[0]], label=f'Reference {monitored_features[0]}')
    sns.kdeplot(data=curr_df[monitored_features[0]], label=f'Current {monitored_features[0]}')

    plt.title(f'KDE Subplot for {monitored_features[0]}')  # Changed plt.set_title to plt.title
    plt.xlabel(monitored_features[0])  # Changed plt.set_xlabel to plt.xlabel
    plt.ylabel('Density')  # Changed plt.set_ylabel to plt.ylabel
    plt.legend()

    plt.tight_layout()
    
    df = df.drop(['year', 'month'], axis=1)

    mlclient.log(operation_type = "register_plots",dbutils = dbutils, figure_to_save=fig, 
        plot_name=f'Store_{store_id}_Item_{monitored_features[0]}_Year_{curr_year}_Month_{curr_month}', 
        folder_name = 'data_drift/LAG_drift_Plots',
        request_type="push_plot")

    return plt

def get_histogram_plot(df,ref_year,ref_month,curr_year,curr_month,monitored_features,date_column=None,store_id=None):

    df[date_column] = pd.to_datetime(df[date_column]) 

    df['year'] = df[date_column].dt.year
    df['month'] = df[date_column].dt.month

    ref_df = df[(df['year'] == ref_year) & (df['month'] == ref_month) ]
    curr_df = df[(df['year'] == curr_year) & (df['month'] == curr_month) ]

    fig = plt.figure(figsize=(10, 6))

    sns.histplot(data=ref_df, x=monitored_features[0], alpha=0.5,color="blue", label=f'Reference {monitored_features[0]}')
    sns.histplot(data=curr_df, x=monitored_features[0], alpha=0.5,color="orange", label=f'Current {monitored_features[0]}')
    plt.title(f'Histogram for {monitored_features[0]}')
    plt.xlabel(monitored_features[0])
    plt.ylabel('Density')
    plt.legend()

    plt.tight_layout()
    
    df = df.drop(['year', 'month'], axis=1)

    mlclient.log(operation_type = "register_plots",dbutils = dbutils, figure_to_save=fig, 
        plot_name=f'Store_{store_id}_Item_{monitored_features[0]}_Year_{curr_year}_Month_{curr_month}', 
        folder_name = 'data_drift/LAG_drift_Plots',
        request_type="push_plot")

    return plt

def fix_df(df):
    
    df=df.drop(["feature_is_drift_list","drifted_features"],axis=1)
    
    df_columns_list=list(df.columns)
    data_list = df.values.tolist()
    final_df=[]

    for  index, row in df.iterrows():
        temp_list=[]
        for col in df.columns :
            # print(col)
            # print(row[col])
            if row[col]>0.5 and col not in ["Batch_No","Current_Data_MONTH","Current_Data_YEAR","Store_No","drifted_features"]:
                substring_to_remove="_weighted_is_drift"
                temp_list.append( col.replace(substring_to_remove, "").strip())

        final_df.append(data_list[index]+[temp_list])


    df_columns_list.append("drifted_features")
    final_df=pd.DataFrame(final_df)
    final_df.columns=df_columns_list

    return pd.DataFrame(final_df)


### Helper method to track which months data is being monitored

In [0]:
def set_up_task_log_table(ref_year,ref_month,curr_year,curr_month):
    """
    Sets up task log table and calls APIs for run and task add
    """
    try:
        # recording timestamp
        ts = str(int(time.time() * 1000000))

        # checking if task log table exists, creating a delta if it does not exist
        if(DeltaTable.isDeltaTable(spark, task_log_path)):
            print("task_log_table existing")
            df_task = spark.read.format("delta").load(task_log_path)
            ref_month = ref_month +1
            curr_month = curr_month + 1
    
            df_record = [(ref_year,ref_month,curr_year,curr_month)]
            input_schema = StructType([
                                 StructField("ref_year", IntegerType(), True), 
                                 StructField("ref_month", IntegerType(), True),
                                 StructField("curr_year", IntegerType(), True),
                                 StructField("curr_month", IntegerType(), True),
                                ])
            new_record_df = spark.createDataFrame(df_record, schema = input_schema)
            new_record_df.write.format("delta").mode("append").save(task_log_path)
            df_task = spark.read.format("delta").load(task_log_path)
            print(df_task.display())
            return df_task

        else:
            print("creating new record in task log table")
            marker = 0
            schema = StructType([
                                 StructField("ref_year", IntegerType(), True), 
                                 StructField("ref_month", IntegerType(), True),
                                 StructField("curr_year", IntegerType(), True),
                                 StructField("curr_month", IntegerType(), True),
                                ])

            df_column_name = ['ref_year', 'ref_month', 'curr_year', 'curr_month']
            #df_record = [("task_-1",ts,date,batch_size, None)]
            df_record = [(ref_year,ref_month,curr_year,curr_month)]
            df_task = spark.createDataFrame(df_record, schema=schema)
            utils.df_write(task_log_path,df_task,"append")
            print(df_task.display())
            return df_task
        
    except Exception as e:
        print(e)
    

In [0]:
# Converting PySpark data into pandas
if mode == 'inference':
    task_log_path = f"dbfs:/mnt/{az_container_name}/{env}/{project_id}/{version}/{str(job_id)}/tasks/{task_key}"
    run_number = int(len(df_pandas)/batch_size)
    if run_number == 1:
        ref_year = 2013
        ref_month = 1
        curr_year = ref_year + 1
        curr_month = ref_month
        df_task_log = set_up_task_log_table(ref_year,ref_month,curr_year,curr_month)
    print(int(run_number))

In [0]:
# Converting PySpark data into pandas
if mode == 'train' or mode == 'preview':
    task_log_path = f"dbfs:/mnt/{az_container_name}/{env}/{project_id}/{version}/{str(job_id)}/tasks/{task_key}"
    run_number =1
    if run_number == 1:
        ref_year = 2013
        ref_month = 2
        curr_year = ref_year + 1
        curr_month = ref_month
        df_task_log = set_up_task_log_table(ref_year,ref_month,curr_year,curr_month)
    print(int(run_number))

if mode == 'train' or mode == 'preview':
    
    pd_df = df.toPandas()
    pd_df.head()
    # Create a dictionary to map old column names to new column names
    new_column_names = {'store': 'store_id', 'item': 'item_id'}
    # Rename the columns using the dictionary
    pd_df.rename(columns=new_column_names, inplace=True)
    drift_df_for_all_stores=pd.DataFrame()
    list_of_drift_df_for_all_stores=[]

    for store_no in sorted(list(pd_df["store_id"].unique())):
    
        store_data = select_store_data(pd_df,store_no,50)
        #run_number = len(pd_df)/batch_size
        #run_number = 1
        # print(int(run_number))
        monitored_features = list(store_data.columns[6:])
        monitoring_algos = ["acf"]

        if run_number == 1:
            result= monitor_data_drift(store_data,ref_year,ref_month,curr_year,curr_month,monitored_features,monitoring_algos,date_column = 'date_LyTg')
            # print(f"DRIFT RESULT: {result}")
            list_of_drift_df_for_all_stores.append(result)

        else:
            df_task_log = spark.read.load(task_log_path)
            df_task_pd = df_task_log.toPandas()
            df_task_pd_final = df_task_pd.sort_values(by=['ref_year','ref_month'], ascending=True)
            last_record = df_task_pd_final.iloc[-1].to_dict()
            ref_year = last_record.get('ref_year')
            ref_month = last_record.get('ref_month')
            curr_year = last_record.get('curr_year')
            curr_month = last_record.get('curr_month')
            print(ref_year,ref_month,curr_year,curr_month)

            result= monitor_data_drift(store_data,ref_year,ref_month,curr_year,curr_month,monitored_features,monitoring_algos,date_column = 'date_LyTg')
            list_of_drift_df_for_all_stores.append(result)
    
    drift_df_for_all_stores=get_item_drift(drift_list=list_of_drift_df_for_all_stores,monitored_features=monitored_features,Batch_No=run_number,Current_Data_MONTH=curr_month,Current_Data_YEAR=curr_year)
    
    drift_df_for_all_stores= fix_df(drift_df_for_all_stores[0])
    fig = plt.figure(figsize=(15, 8))

    sns_df=drift_df_for_all_stores.drop(["Batch_No","Current_Data_MONTH","Store_No","Current_Data_YEAR","drifted_features"],axis=1)
    # Create a dictionary to store the mapping between original and sliced names
    name_mapping = {}
    # Loop through each column name, slice the name, and update the mapping
    for column_name in sns_df.columns:
        sliced_name = column_name[0:10]
        name_mapping[column_name] = sliced_name

    # Rename the columns using the mapping
    sns_df.rename(columns=name_mapping, inplace=True)
    sns.heatmap(sns_df,cmap=sns.cubehelix_palette(as_cmap=True))

    plt.show()
    mlclient.log(operation_type = "register_plots",dbutils = dbutils, figure_to_save=fig, 
        plot_name='LAG_DRIFT_SummaryHeatMap_'+str(curr_year)+'_'+str(curr_month),
        folder_name = 'data_drift',
        request_type="push_plot")
    time.sleep(20)
    print('sleep_done')
    list_of_item_drifted_with_store_no=get_item_drifted_with_store_no(drift_df_for_all_stores)

    if len(list_of_item_drifted_with_store_no)==0 :
        #if nothing drifted in any item sales for any_store  

        for init_feature in ["item_id_1_sales","item_id_2_sales"]:
            plot1=plot_acf_subplots(store_data,ref_year,ref_month,curr_year,curr_month,[init_feature],date_column="date_LyTg")
            get_kde_plot(store_data,ref_year,ref_month,curr_year,curr_month,[init_feature],date_column="date_LyTg")
            get_histogram_plot(store_data,ref_year,ref_month,curr_year,curr_month,[init_feature],date_column="date_LyTg")
    else:
        for item_with_store_no in list_of_item_drifted_with_store_no[:3]:
            print(item_with_store_no)
            store_data = select_store_data(pd_df,item_with_store_no[0],50)
            plot1=plot_acf_subplots(store_data,ref_year,ref_month,curr_year,curr_month,[item_with_store_no[1]],date_column="date_LyTg",store_id=item_with_store_no[0])
            get_kde_plot(store_data,ref_year,ref_month,curr_year,curr_month,[item_with_store_no[1]],date_column="date_LyTg",store_id=item_with_store_no[0])
            get_histogram_plot(store_data,ref_year,ref_month,curr_year,curr_month,[item_with_store_no[1]],date_column="date_LyTg",store_id=item_with_store_no[0])

    if ref_month == 12 and curr_month == 12:
        ref_month = 0
        ref_year = ref_year + 1
        curr_month = 0
        curr_year = curr_year + 1
    df_task_log = set_up_task_log_table(ref_year,ref_month,curr_year,curr_month)



### Main business logic

In [0]:
if mode == 'inference':
    
    pd_df = df.toPandas()
    pd_df.head()
    # Create a dictionary to map old column names to new column names
    new_column_names = {'store': 'store_id', 'item': 'item_id'}
    # Rename the columns using the dictionary
    pd_df.rename(columns=new_column_names, inplace=True)
    drift_df_for_all_stores=pd.DataFrame()
    list_of_drift_df_for_all_stores=[]

    for store_no in sorted(list(pd_df["store_id"].unique())):
    
        store_data = select_store_data(pd_df,store_no,50)
        #run_number = len(pd_df)/batch_size
        #run_number = 1
        # print(int(run_number))
        monitored_features = list(store_data.columns[6:])
        monitoring_algos = ["acf"]

        if run_number == 1:
            result= monitor_data_drift(store_data,ref_year,ref_month,curr_year,curr_month,monitored_features,monitoring_algos,date_column = 'date_LyTg')
            # print(f"DRIFT RESULT: {result}")
            list_of_drift_df_for_all_stores.append(result)

        else:
            df_task_log = spark.read.load(task_log_path)
            df_task_pd = df_task_log.toPandas()
            df_task_pd_final = df_task_pd.sort_values(by=['ref_year','ref_month'], ascending=True)
            last_record = df_task_pd_final.iloc[-1].to_dict()
            ref_year = last_record.get('ref_year')
            ref_month = last_record.get('ref_month')
            curr_year = last_record.get('curr_year')
            curr_month = last_record.get('curr_month')
            print(ref_year,ref_month,curr_year,curr_month)

            result= monitor_data_drift(store_data,ref_year,ref_month,curr_year,curr_month,monitored_features,monitoring_algos,date_column = 'date_LyTg')
            list_of_drift_df_for_all_stores.append(result)
    
    drift_df_for_all_stores=get_item_drift(drift_list=list_of_drift_df_for_all_stores,monitored_features=monitored_features,Batch_No=run_number,Current_Data_MONTH=curr_month,Current_Data_YEAR=curr_year)
    
    drift_df_for_all_stores= fix_df(drift_df_for_all_stores[0])
    fig = plt.figure(figsize=(15, 8))

    sns_df=drift_df_for_all_stores.drop(["Batch_No","Current_Data_MONTH","Store_No","Current_Data_YEAR","drifted_features"],axis=1)
    # Create a dictionary to store the mapping between original and sliced names
    name_mapping = {}
    # Loop through each column name, slice the name, and update the mapping
    for column_name in sns_df.columns:
        sliced_name = column_name[0:10]
        name_mapping[column_name] = sliced_name

    # Rename the columns using the mapping
    sns_df.rename(columns=name_mapping, inplace=True)
    sns.heatmap(sns_df,cmap=sns.cubehelix_palette(as_cmap=True))

    plt.show()
    mlclient.log(operation_type = "register_plots",dbutils = dbutils, figure_to_save=fig, 
        plot_name='LAG_DRIFT_SummaryHeatMap_'+str(curr_year)+'_'+str(curr_month),
        folder_name = 'data_drift',
        request_type="push_plot")
    time.sleep(20)
    print('sleep_done')
    list_of_item_drifted_with_store_no=get_item_drifted_with_store_no(drift_df_for_all_stores)

    if len(list_of_item_drifted_with_store_no)==0 :
        #if nothing drifted in any item sales for any_store  

        for init_feature in ["item_id_1_sales","item_id_2_sales"]:
            plot1=plot_acf_subplots(store_data,ref_year,ref_month,curr_year,curr_month,[init_feature],date_column="date_LyTg")
            get_kde_plot(store_data,ref_year,ref_month,curr_year,curr_month,[init_feature],date_column="date_LyTg")
            get_histogram_plot(store_data,ref_year,ref_month,curr_year,curr_month,[init_feature],date_column="date_LyTg")
    else:
        for item_with_store_no in list_of_item_drifted_with_store_no[:3]:
            print(item_with_store_no)
            store_data = select_store_data(pd_df,item_with_store_no[0],50)
            plot1=plot_acf_subplots(store_data,ref_year,ref_month,curr_year,curr_month,[item_with_store_no[1]],date_column="date_LyTg",store_id=item_with_store_no[0])
            get_kde_plot(store_data,ref_year,ref_month,curr_year,curr_month,[item_with_store_no[1]],date_column="date_LyTg",store_id=item_with_store_no[0])
            get_histogram_plot(store_data,ref_year,ref_month,curr_year,curr_month,[item_with_store_no[1]],date_column="date_LyTg",store_id=item_with_store_no[0])

    if ref_month == 12 and curr_month == 12:
        ref_month = 0
        ref_year = ref_year + 1
        curr_month = 0
        curr_year = curr_year + 1
    df_task_log = set_up_task_log_table(ref_year,ref_month,curr_year,curr_month)


In [0]:
checkpoint_path = mlclient.log(operation_type = "register_plots",
    dbutils = dbutils, 
    request_type="fp_checkpoint")
artifact_path = f"{checkpoint_path}/intermediate_checkpoint"
df_final.write.format('delta').mode('overwrite').save(artifact_path)
print(f'The artifact has been posted successfully at {artifact_path}')

<b> Saving the result of the Custom Transformation

The final dataframe can be saved using the method <b>save_result_df_and_exit</b>. The result is stored and passed on to the downstream nodes, if any. Once the data is saved, the notebook exits.

In [0]:
# Saving resultant data using helper method save_result_df_and_exit
mlclient.log(operation_type = "save_result_df_and_exit",
    df=df_final,
    markers = {},
    dbutils=dbutils,
    spark=spark,
    primary_keys=[])